In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install nilearn ## for fMRI

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 123.7 MB/s eta 0:00:00


In [ ]:
!pip install nibabel

In [ ]:
!pip install spektral

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 10.5 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
from nilearn import plotting, datasets, image, masking
import nibabel as nib
import matplotlib.pyplot as plt
import os
from glob import glob

In [ ]:
##-------------------ROI features

#path = '/content/drive/My Drive/ICASSP_2025/Perkinson_Topo/'
#path = '/content/drive/My Drive/MICCAI_2024/EEG_ADHD_Sweden/Dataset/'
path_AD_SLD= '/content/drive/My Drive/AAAI_fMRI_2025/Final_ROI_SchaeferAtlas/ADHD/SLD/'
path_AD_SSD= '/content/drive/My Drive/AAAI_fMRI_2025/Final_ROI_SchaeferAtlas/ADHD/SSD/'
path_AD_SSI= '/content/drive/My Drive/AAAI_fMRI_2025/Final_ROI_SchaeferAtlas/ADHD/SSI/'
path_AD_VLD= '/content/drive/My Drive/AAAI_fMRI_2025/Final_ROI_SchaeferAtlas/ADHD/VLD/'

path_HC_SLD= '/content/drive/My Drive/AAAI_fMRI_2025/Final_ROI_SchaeferAtlas/Non-ADHD/SLD/'
path_HC_SSD= '/content/drive/My Drive/AAAI_fMRI_2025/Final_ROI_SchaeferAtlas/Non-ADHD/SSD/'
path_HC_SSI= '/content/drive/My Drive/AAAI_fMRI_2025/Final_ROI_SchaeferAtlas/Non-ADHD/SSI/'
path_HC_VLD= '/content/drive/My Drive/AAAI_fMRI_2025/Final_ROI_SchaeferAtlas/Non-ADHD/VLD/'


In [ ]:
## 1. Combine all subjcts ROI Feature matrix in a single file - Ex: ROIFeatures_All_SLD_ADHD.csv (combination of all ADHD subjects' ROI features (SLD))
import pandas as pd
import numpy as np
import os
from glob import glob

# Define all folders
task_paths = {
    "ADHD": {
        "SLD": "/content/drive/My Drive/AAAI_fMRI_2025/AAAI_Final_ROI_Yeo17Atlas/ROI_features/ADHD/SLD/",
        "SSD": "/content/drive/My Drive/AAAI_fMRI_2025/AAAI_Final_ROI_Yeo17Atlas/ROI_features/ADHD/SSD/",
        "SSI": "/content/drive/My Drive/AAAI_fMRI_2025/AAAI_Final_ROI_Yeo17Atlas/ROI_features/ADHD/SSI/",
        "VLD": "/content/drive/My Drive/AAAI_fMRI_2025/AAAI_Final_ROI_Yeo17Atlas/ROI_features/ADHD/VLD/"
    },
    "HC": {
        "SLD": "/content/drive/My Drive/AAAI_fMRI_2025/AAAI_Final_ROI_Yeo17Atlas/ROI_features/HC/SLD/",
        "SSD": "/content/drive/My Drive/AAAI_fMRI_2025/AAAI_Final_ROI_Yeo17Atlas/ROI_features/HC/SSD/",
        "SSI": "/content/drive/My Drive/AAAI_fMRI_2025/AAAI_Final_ROI_Yeo17Atlas/ROI_features/HC/SSI/",
        #"VLD": "/content/drive/My Drive/AAAI_fMRI_2025/Final_ROI_SchaeferAtlas/Non-ADHD/VLD/"
        "VLD": "/content/drive/My Drive/AAAI_fMRI_2025/AAAI_Final_ROI_Yeo17Atlas/ROI_features/HC/VLD/"
    }
}

# Destination directory
output_base = "/content/drive/My Drive/AAAI_fMRI_2025/AAAI_Final_ROI_Yeo17Atlas/ROI_features"

# Store DataFrame shapes
results = []

# Traverse each path and save combined matrices
for label, task_dirs in task_paths.items():
    for task, folder in task_dirs.items():
        data = []
        csv_files = glob(os.path.join(folder, "*.csv"))
        for file_path in csv_files:
            try:
                subject_id = os.path.basename(file_path).split("_")[0]
                df = pd.read_csv(file_path)
                roi_values = df["MeanActivation"].values
                if len(roi_values) == 17 and not np.all(roi_values == 0):
                    data.append([subject_id] + roi_values.tolist())
            except Exception as e:
                print(f"Error reading {file_path}: {e}")

        if data:
            columns = ["SubjectID"] + [f"ROI_{i+1}" for i in range(17)]
            matrix_df = pd.DataFrame(data, columns=columns)
            filename = f"AllROIFeatures_{task}_{label}.csv"
            save_path = os.path.join(output_base, filename)
            matrix_df.to_csv(save_path, index=False)
            results.append((filename, matrix_df.shape))

# Display shapes
results_df = pd.DataFrame(results, columns=["File", "Shape"])
print(results_df.columns)
print(results_df.head())
#import ace_tools as tools; tools.display_dataframe_to_user(name="Saved Feature Matrix Sizes", dataframe=results_df)


In [ ]:
##2. ROI Time series from features--Yeo17 atlas
import pandas as pd
import numpy as np
import os
from glob import glob
import nibabel as nib
from nilearn.image import resample_to_img

# === Paths ===
original_atlas_path = "/content/drive/My Drive/AAAI_fMRI_2025/AAAI_Final_ROI_Yeo17Atlas/Yeo2011_17Networks_MNI152_FreeSurferConformed1mm.nii"
selected_roi_dir = "/content/drive/My Drive/AAAI_fMRI_2025/AAAI_Final_ROI_Yeo17Atlas/ROI_features"
fmri_base_dir = "/content/drive/My Drive/AAAI_fMRI_2025/Func_4D_images"
output_base = "/content/drive/My Drive/AAAI_fMRI_2025/AAAI_Final_ROI_Yeo17Atlas/ROI_TimeSeries"
os.makedirs(output_base, exist_ok=True)

# === Helper: Load selected ROIs (excluding SubjectID) ===
def load_selected_rois(task, group):
    csv_path = os.path.join(selected_roi_dir, f"AllROIFeatures_{task}_{group}.csv")
    df = pd.read_csv(csv_path)
    roi_columns = [col for col in df.columns if col.startswith("ROI_")]
    return roi_columns

# === Settings ===
tasks = ["SLD", "SSD", "SSI", "VLD"]
groups = ["ADHD", "HC"]

# === Main Loop ===
for group in groups:
    group_output_dir = os.path.join(output_base, group)
    os.makedirs(group_output_dir, exist_ok=True)

    for task in tasks:
        selected_rois = load_selected_rois(task, group)
        roi_ids = [int(roi.split("_")[1]) for roi in selected_rois]

        subject_files = sorted(glob(os.path.join(fmri_base_dir, group, task, "swausub-*.nii")))

        for file_path in subject_files:
            try:
                fmri_img = nib.load(file_path)
                fmri_data = fmri_img.get_fdata()
                n_timepoints = fmri_data.shape[-1]

                resampled_atlas = resample_to_img(original_atlas_path, fmri_img, interpolation='nearest')
                atlas_data = resampled_atlas.get_fdata()

                subject_matrix = []
                for roi in roi_ids:
                    roi_mask = (atlas_data == roi)
                    if not np.any(roi_mask):
                        roi_timeseries = np.zeros(n_timepoints)
                    else:
                        voxels_flat = fmri_data.reshape(-1, n_timepoints)
                        mask_flat = roi_mask.flatten()
                        roi_voxels = voxels_flat[mask_flat, :]
                        roi_timeseries = np.mean(roi_voxels, axis=0)
                    subject_matrix.append(roi_timeseries)

                subject_matrix = np.array(subject_matrix).T  # shape: time x ROIs

                sub_id = os.path.basename(file_path).split("_")[0]
                out_file = f"{sub_id}_{task}_{group}_timeseries.csv"
                out_csv = os.path.join(group_output_dir, out_file)

                pd.DataFrame(subject_matrix, columns=selected_rois).to_csv(out_csv, index=False)
                print(f"Saved: {out_csv}")

            except Exception as e:
                print(f"Error processing {file_path}: {e}")


In [ ]:
## 3. Connectivity Matrices (Correlation, Granger Causality, Coherence, Mutual Information) from the ROI time series

import os
import pandas as pd
import numpy as np
from glob import glob
from statsmodels.tsa.stattools import grangercausalitytests
from scipy.signal import coherence
from sklearn.feature_selection import mutual_info_regression

# === Paths ===
base_path = "/content/drive/My Drive/AAAI_fMRI_2025/AAAI_Final_ROI_Yeo17Atlas/ROI_TimeSeries"
output_path = "/content/drive/My Drive/AAAI_fMRI_2025/AAAI_Final_ROI_Yeo17Atlas/Connectivity_Matrices"

groups = ["ADHD", "HC"]
tasks = ["SLD", "SSD", "SSI", "VLD"]
gc_maxlag = 1  # Granger causality lag

# === Ensure group-specific folders exist ===
for group in groups:
    os.makedirs(os.path.join(output_path, group), exist_ok=True)

# === Helper: Compute directional Granger causality matrix ===
def compute_granger_matrix(data, maxlag=1):
    n = data.shape[1]
    gc_matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            try:
                result = grangercausalitytests(data[:, [j, i]], maxlag=maxlag, verbose=False)
                p_val = result[maxlag][0]['ssr_ftest'][1]
                score = -np.log10(p_val + 1e-10)
                gc_matrix[i, j] = score
            except:
                gc_matrix[i, j] = 0
    max_val = np.max(gc_matrix)
    return gc_matrix / max_val if max_val > 0 else gc_matrix

# === Helper: Compute coherence matrix ===
def compute_coherence_matrix(data, fs=1.0):
    n = data.shape[1]
    coh_matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if i != j:
                f, Cxy = coherence(data[:, i], data[:, j], fs=fs)
                coh_matrix[i, j] = np.mean(Cxy)
    return coh_matrix

# === Helper: Compute Mutual Information matrix ===
def compute_mutual_information_matrix(data):
    n = data.shape[1]
    mi_matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if i != j:
                mi_matrix[i, j] = mutual_info_regression(data[:, i].reshape(-1, 1), data[:, j])[0]
    max_val = np.max(mi_matrix)
    return mi_matrix / max_val if max_val > 0 else mi_matrix

# === Process time series and save connectivity matrices ===
connectivity_log = []

for group in groups:
    group_input_dir = os.path.join(base_path, group)
    group_output_dir = os.path.join(output_path, group)

    files = sorted(glob(os.path.join(group_input_dir, "*.csv")))
    for file_path in files:
        try:
            df = pd.read_csv(file_path)
            data = df.values

            # Correlation matrix
            corr = np.corrcoef(data.T)
            corr[np.abs(corr) < 0.5] = 0
            np.fill_diagonal(corr, 0)

            # Granger causality matrix
            gc = compute_granger_matrix(data, maxlag=gc_maxlag)

            # Coherence matrix
            coherence_matrix = compute_coherence_matrix(data)

            # Mutual Information matrix
            mi_matrix = compute_mutual_information_matrix(data)

            base_name = os.path.splitext(os.path.basename(file_path))[0]
            np.save(os.path.join(group_output_dir, f"{base_name}_corr.npy"), corr)
            np.save(os.path.join(group_output_dir, f"{base_name}_gc.npy"), gc)
            np.save(os.path.join(group_output_dir, f"{base_name}_coherence.npy"), coherence_matrix)
            np.save(os.path.join(group_output_dir, f"{base_name}_mi.npy"), mi_matrix)

            connectivity_log.append((base_name, corr.shape[0]))

        except Exception as e:
            print(f"Error processing {file_path}: {e}")
            connectivity_log.append((file_path, str(e)))

# Show log summary
print("Connectivity matrices generation complete.")
print("Sample log:", connectivity_log[:5])


### **TASK Aware Learning (3 cases)-- Multiple classification: Task and Disorder**


In [ ]:
## 1. Data preparation from connectivity matrices--New revided data size (without fusing all connectiviyu matrices)-- size- 30x16x289
import numpy as np
import glob, os

data_dir = '/content/drive/My Drive/AAAI_fMRI_2025/AAAI_Final_ROI_Yeo17Atlas/Connectivity_Matrices'
groups = ['ADHD', 'HC']
tasks = ['SLD', 'SSD', 'SSI', 'VLD']
connectivity_types = ['corr', 'gc', 'coherence', 'mi']

# Subjects explicitly
ADHD_subjects = ["sub-06", "sub-09", "sub-10", "sub-11", "sub-12", "sub-13",
                 "sub-17", "sub-29", "sub-30", "sub-32", "sub-33", "sub-44",
                 "sub-54", "sub-57", "sub-66"]

HC_subjects = ["sub-03", "sub-15", "sub-16", "sub-18", "sub-19", "sub-20",
               "sub-21", "sub-22", "sub-23", "sub-26", "sub-27", "sub-28",
               "sub-31", "sub-35", "sub-46"]

all_subjects = ADHD_subjects + HC_subjects
X = []  # explicitly initialized list

for subj in all_subjects:
    subject_features = []
    group = 'ADHD' if subj in ADHD_subjects else 'HC'

    for task in tasks:
        for conn_type in connectivity_types:
            filename = f'swau{subj}_{task}_{group}_timeseries_{conn_type}.npy'
            filepath = os.path.join(data_dir, group, filename)
            if not os.path.exists(filepath):
                raise FileNotFoundError(f"File not found explicitly: {filepath}")

            matrix = np.load(filepath).flatten()  # explicitly 17×17 flattened to 289
            subject_features.append(matrix)

    X.append(subject_features)

# explicitly convert to numpy array explicitly
X = np.array(X)

print("Explicitly generated data shape:", X.shape)
# Expected explicitly: (30 subjects, 16 features, 289 feature size explicitly) ## #Subjects, #featurs/subject, feature-size


Explicitly generated data shape: (30, 16, 289)


In [ ]:
### 2. Adjacency matrix creation from connectivity matrics
import os
import numpy as np
from glob import glob

connectivity_dir_ADHD = '/content/drive/My Drive/AAAI_fMRI_2025/AAAI_Final_ROI_Yeo17Atlas/Connectivity_Matrices/ADHD'
connectivity_dir_HC = '/content/drive/My Drive/AAAI_fMRI_2025/AAAI_Final_ROI_Yeo17Atlas/Connectivity_Matrices/HC'

ADHD_adj_dir = '/content/drive/My Drive/AAAI_fMRI_2025/AAAI_Final_ROI_Yeo17Atlas/Graph_Adjacency_Matrices/ADHD'
HC_adj_dir = '/content/drive/My Drive/AAAI_fMRI_2025/AAAI_Final_ROI_Yeo17Atlas/Graph_Adjacency_Matrices/HC'

os.makedirs(ADHD_adj_dir, exist_ok=True)
os.makedirs(HC_adj_dir, exist_ok=True)

connectivity_types = ['corr', 'gc', 'coherence', 'mi']

def matrix_to_adjacency(matrix, threshold=0.2):
    adjacency = np.abs(matrix)
    adjacency[adjacency < threshold] = 0
    np.fill_diagonal(adjacency, 0)
    return adjacency

def process_and_save_adj_matrices(input_dir, output_dir, threshold=0.2):
    npy_files = sorted(glob(os.path.join(input_dir, '*.npy')))
    saved = 0
    for file in npy_files:
        matrix = np.load(file)
        adj_matrix = matrix_to_adjacency(matrix, threshold)

        # Ensure there are edges after thresholding
        if np.sum(adj_matrix) == 0:
            print(f"No edges found for {file} at threshold {threshold}. Consider lowering threshold.")
            continue

        # Save explicitly
        filename = os.path.basename(file).replace('.npy', '_adjacency.npy')
        save_path = os.path.join(output_dir, filename)
        np.save(save_path, adj_matrix)
        saved += 1

    print(f"Adjacency matrices saved in {output_dir}: {saved}")

# Recreate adjacency matrices with lower threshold explicitly
process_and_save_adj_matrices(connectivity_dir_ADHD, ADHD_adj_dir, threshold=0.2)
process_and_save_adj_matrices(connectivity_dir_HC, HC_adj_dir, threshold=0.2)


In [ ]:
#!pip install --upgrade networkx

In [ ]:
# ## 3. Graph generation from adjacency matrices

import os
import numpy as np
import networkx as nx
import pickle
from glob import glob

# Directories
connectivity_dir_ADHD = '/content/drive/My Drive/AAAI_fMRI_2025/AAAI_Final_ROI_Yeo17Atlas/Connectivity_Matrices/ADHD'
connectivity_dir_HC = '/content/drive/My Drive/AAAI_fMRI_2025/AAAI_Final_ROI_Yeo17Atlas/Connectivity_Matrices/HC'

adjacency_dir_ADHD = '/content/drive/My Drive/AAAI_fMRI_2025/AAAI_Final_ROI_Yeo17Atlas/Graph_Adjacency_Matrices/ADHD'
adjacency_dir_HC = '/content/drive/My Drive/AAAI_fMRI_2025/AAAI_Final_ROI_Yeo17Atlas/Graph_Adjacency_Matrices/HC'

graph_output_dir_ADHD = '/content/drive/My Drive/AAAI_fMRI_2025/AAAI_Final_ROI_Yeo17Atlas/Graphs/ADHD'
graph_output_dir_HC = '/content/drive/My Drive/AAAI_fMRI_2025/AAAI_Final_ROI_Yeo17Atlas/Graphs/HC'

# Ensure output directories exist
os.makedirs(graph_output_dir_ADHD, exist_ok=True)
os.makedirs(graph_output_dir_HC, exist_ok=True)

connectivity_types = ['corr', 'gc', 'coherence', 'mi']

# Function to load matrix explicitly
def load_matrix(filepath):
    return np.load(filepath)

# Function to create graph from adjacency and connectivity matrices
def create_weighted_graph(adjacency, connectivity):
    G = nx.Graph()
    num_nodes = adjacency.shape[0]

    # Add nodes explicitly
    for node in range(num_nodes):
        G.add_node(node)

    # Add edges with original connectivity values as weights explicitly
    for i in range(num_nodes):
        for j in range(i + 1, num_nodes):
            if adjacency[i, j] > 0:  # Edge exists explicitly according to adjacency matrix
                G.add_edge(i, j, weight=connectivity[i, j])

    return G

# Function explicitly defined to process and save graphs
def process_group(connectivity_dir, adjacency_dir, graph_output_dir, group_name):
    adjacency_files = sorted(glob(os.path.join(adjacency_dir, '*_adjacency.npy')))

    if not adjacency_files:
        print(f"No adjacency files found in {adjacency_dir}")
        return

    saved_count = 0

    for adj_path in adjacency_files:
        try:
            # Load adjacency matrix explicitly
            adjacency = load_matrix(adj_path)

            # Extract file basename explicitly to find corresponding connectivity matrix
            base_filename = os.path.basename(adj_path).replace('_adjacency.npy', '')

            # Determine connectivity type from filename
            connectivity_type = None
            for ctype in connectivity_types:
                if ctype in base_filename:
                    connectivity_type = ctype
                    break
            if connectivity_type is None:
                print(f"Connectivity type not found for {base_filename}")
                continue

            # Construct connectivity matrix filename
            connectivity_file = f"{base_filename}.npy"
            connectivity_path = os.path.join(connectivity_dir, connectivity_file)

            if not os.path.exists(connectivity_path):
                print(f"Connectivity file not found: {connectivity_path}")
                continue

            connectivity_matrix = load_matrix(connectivity_path)

            # Create the weighted graph explicitly
            graph = create_weighted_graph(adjacency, connectivity_matrix)

            # Save the graph explicitly using pickle
            graph_filename = f"{base_filename}_graph.pkl"
            graph_save_path = os.path.join(graph_output_dir, graph_filename)
            with open(graph_save_path, 'wb') as f:
                pickle.dump(graph, f)

            saved_count += 1

        except Exception as e:
            print(f"Error processing {adj_path}: {e}")

    print(f"Total graphs created and saved for {group_name}: {saved_count}")

# Explicitly process ADHD group
process_group(connectivity_dir_ADHD, adjacency_dir_ADHD, graph_output_dir_ADHD, "ADHD")

# Explicitly process HC group
process_group(connectivity_dir_HC, adjacency_dir_HC, graph_output_dir_HC, "HC")

print("\nGraph creation from adjacency and connectivity matrices completed successfully.")


In [ ]:
## 4. Graph Auhmentation--VAE-GAN Functional brain network identification and fMRI augmentation using a ##VAE-GAN framework(Review: IEEE Acces paper)
import os
import numpy as np
import networkx as nx
import pickle
from glob import glob
import random
import re

# Set explicit random seed for reproducibility
np.random.seed(42)
random.seed(42)

# Directories explicitly
ADHD_graph_dir = '/content/drive/My Drive/AAAI_fMRI_2025/AAAI_Final_ROI_Yeo17Atlas/Graphs/ADHD'
HC_graph_dir = '/content/drive/My Drive/AAAI_fMRI_2025/AAAI_Final_ROI_Yeo17Atlas/Graphs/HC'

ADHD_aug_dir = '/content/drive/My Drive/AAAI_fMRI_2025/AAAI_Final_ROI_Yeo17Atlas/Graph_Augmentation/ADHD'
HC_aug_dir = '/content/drive/My Drive/AAAI_fMRI_2025/AAAI_Final_ROI_Yeo17Atlas/Graph_Augmentation/HC'

os.makedirs(ADHD_aug_dir, exist_ok=True)
os.makedirs(HC_aug_dir, exist_ok=True)

# Load explicitly original graphs only
def load_original_graphs(graph_dir):
    graph_files = sorted(glob(os.path.join(graph_dir, '*.pkl')))
    original_graphs = []
    # Pattern explicitly matches original graphs only
    pattern = re.compile(r'swausub-\d+_[A-Za-z]+_(ADHD|HC)_timeseries_[a-z]+_graph\.pkl$')
    for file_path in graph_files:
        filename = os.path.basename(file_path)
        if pattern.match(filename):
            with open(file_path, 'rb') as f:
                graph = pickle.load(f)
                original_graphs.append((filename, graph))
    return original_graphs

# Graph augmentation explicitly defined
def augment_graph(graph, perturbation_rate=0.05, dropout_rate=0.05):
    G_aug = graph.copy()
    for u, v, data in G_aug.edges(data=True):
        original_weight = data.get('weight', 0)
        noise = np.random.normal(0, perturbation_rate * original_weight)
        data['weight'] = max(0, original_weight + noise)
    edges_list = list(G_aug.edges())
    num_edges_remove = int(dropout_rate * len(edges_list))
    if num_edges_remove > 0:
        edges_remove = random.sample(edges_list, num_edges_remove)
        G_aug.remove_edges_from(edges_remove)
    return G_aug

# Function explicitly processing and augmenting graphs
def process_and_augment_graphs(input_dir, output_dir, group_name, augmentation_factor=3):
    graphs = load_original_graphs(input_dir)
    total_augmented = 0
    if not graphs:
        print(f"No original graphs found in {input_dir}")
        return
    for file_name, graph in graphs:
        base_name = file_name.replace('.pkl', '')
        for aug_idx in range(augmentation_factor):
            augmented_graph = augment_graph(graph)
            augmented_filename = f"{base_name}_aug{aug_idx+1}.pkl"
            save_path = os.path.join(output_dir, augmented_filename)
            with open(save_path, 'wb') as f:
                pickle.dump(augmented_graph, f)
            total_augmented += 1
    print(f"Total augmented graphs generated for {group_name}: {total_augmented}")

# Run explicitly for ADHD and HC
process_and_augment_graphs(ADHD_graph_dir, ADHD_aug_dir, "ADHD", augmentation_factor=7) ## change the augmented graphs using 'augmentation_factor'=7
process_and_augment_graphs(HC_graph_dir, HC_aug_dir, "HC", augmentation_factor=7)

print("\nGraph augmentation completed successfully.")


In [ ]:
## 5. Triplet generation--Task and Disorder
import os
import numpy as np
import pickle
from glob import glob
import random

# Seed for reproducibility explicitly
random.seed(42)
np.random.seed(42)

# Paths explicitly
ADHD_graph_dir = '/content/drive/My Drive/AAAI_fMRI_2025/AAAI_Final_ROI_Yeo17Atlas/Graphs/ADHD'
HC_graph_dir = '/content/drive/My Drive/AAAI_fMRI_2025/AAAI_Final_ROI_Yeo17Atlas/Graphs/HC'
ADHD_aug_dir = '/content/drive/My Drive/AAAI_fMRI_2025/AAAI_Final_ROI_Yeo17Atlas/Graph_Augmentation/ADHD'
HC_aug_dir = '/content/drive/My Drive/AAAI_fMRI_2025/AAAI_Final_ROI_Yeo17Atlas/Graph_Augmentation/HC'

# Combine original and augmented graphs explicitly
def load_graphs(graph_dirs):
    graphs = []
    for graph_dir in graph_dirs:
        files = sorted(glob(os.path.join(graph_dir, '*.pkl')))
        for file in files:
            with open(file, 'rb') as f:
                graph = pickle.load(f)
            fname = os.path.basename(file)
            parts = fname.split('_')
            subject_id = parts[0]  # e.g., swausub-06
            task = parts[1]        # e.g., SLD
            group = 'ADHD' if 'ADHD' in fname else 'HC'
            connectivity_type = fname.split('_')[-2]  # e.g., corr, gc, coherence, mi
            graphs.append({
                'graph': graph,
                'subject': subject_id,
                'task': task,
                'group': group,
                'connectivity': connectivity_type,
                'filename': fname
            })
    return graphs

# Explicitly load all graphs
ADHD_graphs = load_graphs([ADHD_graph_dir, ADHD_aug_dir])
HC_graphs = load_graphs([HC_graph_dir, HC_aug_dir])
all_graphs = ADHD_graphs + HC_graphs

print(f"Total ADHD graphs: {len(ADHD_graphs)}")
print(f"Total HC graphs: {len(HC_graphs)}")
print(f"Total graphs combined: {len(all_graphs)}")

# ===== Explicit Task Triplet Generation =====
task_triplets = []

# Create anchor-positive pairs (same task, same group, different subject)
for anchor in all_graphs:
    anchor_task = anchor['task']
    anchor_group = anchor['group']
    anchor_subject = anchor['subject']

    # Positive candidates: same task, same group, different subject
    positive_candidates = [
        g for g in all_graphs
        if (g['task'] == anchor_task and g['group'] == anchor_group and g['subject'] != anchor_subject)
    ]
    if not positive_candidates:
        continue
    positive = random.choice(positive_candidates)

    # Negative candidates: different task, same group, same subject
    negative_candidates = [
        g for g in all_graphs
        if (g['task'] != anchor_task and g['group'] == anchor_group and g['subject'] == anchor_subject)
    ]
    if not negative_candidates:
        continue
    negative = random.choice(negative_candidates)

    task_triplets.append((anchor, positive, negative))

print(f"Total Task Triplets Generated: {len(task_triplets)}")

# ===== Explicit Disorder Triplet Generation =====
disorder_triplets = []

# Create anchor-positive pairs (same group)
for anchor in all_graphs:
    anchor_group = anchor['group']

    # Positive candidates: same group (any task, any subject excluding anchor itself)
    positive_candidates = [
        g for g in all_graphs
        if (g['group'] == anchor_group and g['filename'] != anchor['filename'])
    ]
    if not positive_candidates:
        continue
    positive = random.choice(positive_candidates)

    # Negative candidates: different group (any task, any subject)
    negative_candidates = [
        g for g in all_graphs
        if g['group'] != anchor_group
    ]
    if not negative_candidates:
        continue
    negative = random.choice(negative_candidates)

    disorder_triplets.append((anchor, positive, negative))

print(f"Total Disorder Triplets Generated: {len(disorder_triplets)}")

# ===== Saving Triplets explicitly =====
triplet_save_dir = '/content/drive/My Drive/AAAI_fMRI_2025/AAAI_Final_ROI_Yeo17Atlas/Graph_Triplets'
os.makedirs(triplet_save_dir, exist_ok=True)

# Explicitly save Task Triplets
task_triplets_path = os.path.join(triplet_save_dir, 'task_triplets.pkl')
with open(task_triplets_path, 'wb') as f:
    pickle.dump(task_triplets, f)
print(f"Task triplets explicitly saved to {task_triplets_path}")

# Explicitly save Disorder Triplets
disorder_triplets_path = os.path.join(triplet_save_dir, 'disorder_triplets.pkl')
with open(disorder_triplets_path, 'wb') as f:
    pickle.dump(disorder_triplets, f)
print(f"Disorder triplets explicitly saved to {disorder_triplets_path}")


In [ ]:
## 6. 3D embeddings with Triplet loss
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Dropout, Layer
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import pickle
import networkx as nx

# Set explicit random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Paths explicitly defined
triplet_dir = '/content/drive/My Drive/AAAI_fMRI_2025/AAAI_Final_ROI_Yeo17Atlas/Graph_Triplets'
task_triplets_path = os.path.join(triplet_dir, 'task_triplets.pkl')
disorder_triplets_path = os.path.join(triplet_dir, 'disorder_triplets.pkl')

# Load triplets explicitly
def load_triplets(filepath):
    with open(filepath, 'rb') as f:
        return pickle.load(f)

task_triplets = load_triplets(task_triplets_path)
disorder_triplets = load_triplets(disorder_triplets_path)

print(f"Task triplets loaded: {len(task_triplets)}")
print(f"Disorder triplets loaded: {len(disorder_triplets)}")

# Convert graphs explicitly to adjacency matrices
def graph_to_adj(graph):
    return nx.to_numpy_array(graph)

# Prepare adjacency matrices explicitly
def prepare_adj_triplets(triplets):
    anchors, positives, negatives = [], [], []
    for anchor, pos, neg in triplets:
        anchors.append(graph_to_adj(anchor['graph']))
        positives.append(graph_to_adj(pos['graph']))
        negatives.append(graph_to_adj(neg['graph']))
    return np.array(anchors), np.array(positives), np.array(negatives)

# Explicitly prepared data
anchor_task, pos_task, neg_task = prepare_adj_triplets(task_triplets)
anchor_dis, pos_dis, neg_dis = prepare_adj_triplets(disorder_triplets)

print("Task data shape:", anchor_task.shape)
print("Disorder data shape:", anchor_dis.shape)

# Parameters explicitly
num_nodes = anchor_task.shape[1]  # 17 ROIs explicitly
node_feature_dim = num_nodes      # Using identity features (one-hot per node explicitly)
embedding_dim = 64                # Final embedding dimension explicitly per node

# Node feature generation layer explicitly defined
class NodeFeatureLayer(Layer):
    def call(self, adj):
        batch_size = tf.shape(adj)[0]
        num_nodes = tf.shape(adj)[1]
        return tf.eye(num_nodes, batch_shape=[batch_size])

# Graph Convolution layer explicitly defined
class GraphConvLayer(Layer):
    def __init__(self, output_dim, activation='relu', **kwargs):
        super().__init__(**kwargs)
        self.output_dim = output_dim
        self.activation = tf.keras.activations.get(activation)

    def build(self, input_shape):
        self.weight = self.add_weight(shape=(input_shape[0][-1], self.output_dim),
                                      initializer='glorot_uniform',
                                      trainable=True)

    def call(self, inputs):
        x, adj = inputs
        h = tf.matmul(x, self.weight)
        output = tf.matmul(adj, h)
        return self.activation(output)

# Create explicitly GCN embedding model without global pooling explicitly (keeps 3D output)
def create_gcn_embedding(num_nodes, embedding_dim):
    adj_input = Input(shape=(num_nodes, num_nodes), name='adjacency_input')
    node_features = NodeFeatureLayer()(adj_input)

    # First GCN layer explicitly
    x = GraphConvLayer(128)([node_features, adj_input])
    x = Dropout(0.5)(x)

    # Second GCN layer explicitly (final node embeddings explicitly)
    x = GraphConvLayer(embedding_dim, activation='linear')([x, adj_input])

    # Explicit output shape: (batch_size, num_nodes, embedding_dim)
    return Model(inputs=adj_input, outputs=x)

# Explicitly create embedding models explicitly (outputting 3D embeddings)
task_embedding_model = create_gcn_embedding(num_nodes, embedding_dim)
disorder_embedding_model = create_gcn_embedding(num_nodes, embedding_dim)

# Explicit triplet loss for 3D embeddings explicitly
def triplet_loss(margin=1.0):
    def loss(_, embeddings):
        anchor, positive, negative = tf.unstack(embeddings, num=3, axis=1)
        # Compute global mean explicitly for triplet comparisons
        anchor_mean = tf.reduce_mean(anchor, axis=1)
        positive_mean = tf.reduce_mean(positive, axis=1)
        negative_mean = tf.reduce_mean(negative, axis=1)

        pos_dist = tf.reduce_sum(tf.square(anchor_mean - positive_mean), axis=1)
        neg_dist = tf.reduce_sum(tf.square(anchor_mean - negative_mean), axis=1)
        return tf.reduce_mean(tf.maximum(pos_dist - neg_dist + margin, 0.0))
    return loss

# Create explicit triplet training model explicitly
def create_triplet_model(embedding_model):
    anchor_input = Input(shape=(num_nodes, num_nodes))
    pos_input = Input(shape=(num_nodes, num_nodes))
    neg_input = Input(shape=(num_nodes, num_nodes))

    anchor_embed = embedding_model(anchor_input)
    pos_embed = embedding_model(pos_input)
    neg_embed = embedding_model(neg_input)

    merged_output = tf.keras.layers.Lambda(lambda x: tf.stack(x, axis=1))(
        [anchor_embed, pos_embed, neg_embed])

    model = Model(inputs=[anchor_input, pos_input, neg_input], outputs=merged_output)
    model.compile(loss=triplet_loss(), optimizer=Adam(1e-4))
    return model

# Explicitly create and train task embedding explicitly
print("\nTraining Task Embedding explicitly:")
task_triplet_model = create_triplet_model(task_embedding_model)
task_triplet_model.fit(
    [anchor_task, pos_task, neg_task],
    np.zeros((anchor_task.shape[0], 3, num_nodes, embedding_dim)),
    epochs=100, batch_size=32,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)]
)

# Explicitly create and train disorder embedding explicitly
print("\nTraining Disorder Embedding explicitly:")
disorder_triplet_model = create_triplet_model(disorder_embedding_model)
disorder_triplet_model.fit(
    [anchor_dis, pos_dis, neg_dis],
    np.zeros((anchor_dis.shape[0], 3, num_nodes, embedding_dim)),
    epochs=100, batch_size=32,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)]
)

# Explicitly save trained embedding models explicitly
model_save_dir = '/content/drive/My Drive/AAAI_fMRI_2025/AAAI_Final_ROI_Yeo17Atlas/GNN_Embeddings'
os.makedirs(model_save_dir, exist_ok=True)

task_embedding_model.save(os.path.join(model_save_dir, 'task_gcn_embedding_model_3d.keras'))
disorder_embedding_model.save(os.path.join(model_save_dir, 'disorder_gcn_embedding_model_3d.keras'))

print("\n3D GNN Embedding models explicitly trained and saved successfully.")


In [ ]:
## TRiplet loss pdf


## 6. 3D embeddings with Triplet loss
import os
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Dropout, Layer
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import pickle
import networkx as nx

# Set explicit random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Paths explicitly defined
triplet_dir = '/content/drive/My Drive/AAAI_fMRI_2025/AAAI_Final_ROI_Yeo17Atlas/Graph_Triplets'
task_triplets_path = os.path.join(triplet_dir, 'task_triplets.pkl')
disorder_triplets_path = os.path.join(triplet_dir, 'disorder_triplets.pkl')

# Load triplets explicitly
def load_triplets(filepath):
    with open(filepath, 'rb') as f:
        return pickle.load(f)

task_triplets = load_triplets(task_triplets_path)
disorder_triplets = load_triplets(disorder_triplets_path)

print(f"Task triplets loaded: {len(task_triplets)}")
print(f"Disorder triplets loaded: {len(disorder_triplets)}")

# Convert graphs explicitly to adjacency matrices
def graph_to_adj(graph):
    return nx.to_numpy_array(graph)

# Prepare adjacency matrices explicitly
def prepare_adj_triplets(triplets):
    anchors, positives, negatives = [], [], []
    for anchor, pos, neg in triplets:
        anchors.append(graph_to_adj(anchor['graph']))
        positives.append(graph_to_adj(pos['graph']))
        negatives.append(graph_to_adj(neg['graph']))
    return np.array(anchors), np.array(positives), np.array(negatives)

# Explicitly prepared data
anchor_task, pos_task, neg_task = prepare_adj_triplets(task_triplets)
anchor_dis, pos_dis, neg_dis = prepare_adj_triplets(disorder_triplets)

print("Task data shape:", anchor_task.shape)
print("Disorder data shape:", anchor_dis.shape)

# Parameters explicitly
num_nodes = anchor_task.shape[1]  # 17 ROIs explicitly
node_feature_dim = num_nodes      # Using identity features (one-hot per node explicitly)
embedding_dim = 64                # Final embedding dimension explicitly per node

# Node feature generation layer explicitly defined
class NodeFeatureLayer(Layer):
    def call(self, adj):
        batch_size = tf.shape(adj)[0]
        num_nodes = tf.shape(adj)[1]
        return tf.eye(num_nodes, batch_shape=[batch_size])

# Graph Convolution layer explicitly defined
class GraphConvLayer(Layer):
    def __init__(self, output_dim, activation='relu', **kwargs):
        super().__init__(**kwargs)
        self.output_dim = output_dim
        self.activation = tf.keras.activations.get(activation)

    def build(self, input_shape):
        self.weight = self.add_weight(shape=(input_shape[0][-1], self.output_dim),
                                      initializer='glorot_uniform',
                                      trainable=True)

    def call(self, inputs):
        x, adj = inputs
        h = tf.matmul(x, self.weight)
        output = tf.matmul(adj, h)
        return self.activation(output)

# Create explicitly GCN embedding model without global pooling explicitly (keeps 3D output)
def create_gcn_embedding(num_nodes, embedding_dim):
    adj_input = Input(shape=(num_nodes, num_nodes), name='adjacency_input')
    node_features = NodeFeatureLayer()(adj_input)

    # First GCN layer explicitly
    x = GraphConvLayer(128)([node_features, adj_input])
    x = Dropout(0.5)(x)

    # Second GCN layer explicitly (final node embeddings explicitly)
    x = GraphConvLayer(embedding_dim, activation='linear')([x, adj_input])

    # Explicit output shape: (batch_size, num_nodes, embedding_dim)
    return Model(inputs=adj_input, outputs=x)

# Explicitly create embedding models explicitly (outputting 3D embeddings)
task_embedding_model = create_gcn_embedding(num_nodes, embedding_dim)
disorder_embedding_model = create_gcn_embedding(num_nodes, embedding_dim)

# Explicit triplet loss for 3D embeddings explicitly
def triplet_loss(margin=1.0):
    def loss(_, embeddings):
        anchor, positive, negative = tf.unstack(embeddings, num=3, axis=1)
        # Compute global mean explicitly for triplet comparisons
        anchor_mean = tf.reduce_mean(anchor, axis=1)
        positive_mean = tf.reduce_mean(positive, axis=1)
        negative_mean = tf.reduce_mean(negative, axis=1)

        pos_dist = tf.reduce_sum(tf.square(anchor_mean - positive_mean), axis=1)
        neg_dist = tf.reduce_sum(tf.square(anchor_mean - negative_mean), axis=1)
        return tf.reduce_mean(tf.maximum(pos_dist - neg_dist + margin, 0.0))
    return loss

# Create explicit triplet training model explicitly
def create_triplet_model(embedding_model):
    anchor_input = Input(shape=(num_nodes, num_nodes))
    pos_input = Input(shape=(num_nodes, num_nodes))
    neg_input = Input(shape=(num_nodes, num_nodes))

    anchor_embed = embedding_model(anchor_input)
    pos_embed = embedding_model(pos_input)
    neg_embed = embedding_model(neg_input)

    merged_output = tf.keras.layers.Lambda(lambda x: tf.stack(x, axis=1))(
        [anchor_embed, pos_embed, neg_embed])

    model = Model(inputs=[anchor_input, pos_input, neg_input], outputs=merged_output)
    model.compile(loss=triplet_loss(), optimizer=Adam(1e-4))
    return model

# === Collect loss history during training ===
# Store the history objects from model.fit()
task_triplet_model = create_triplet_model(task_embedding_model)
task_history = task_triplet_model.fit(
    [anchor_task, pos_task, neg_task],
    np.zeros((anchor_task.shape[0], 3, num_nodes, embedding_dim)),
    epochs=200, batch_size=64,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)]
)

# Explicitly create and train disorder embedding explicitly
print("\nTraining Disorder Embedding explicitly:")
disorder_triplet_model = create_triplet_model(disorder_embedding_model)
disorder_history= disorder_triplet_model.fit(
    [anchor_dis, pos_dis, neg_dis],
    np.zeros((anchor_dis.shape[0], 3, num_nodes, embedding_dim)),
    epochs=200, batch_size=64,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)]
)

# Explicitly save trained embedding models explicitly
model_save_dir = '/content/drive/My Drive/AAAI_fMRI_2025/AAAI_Final_ROI_Yeo17Atlas/GNN_Embeddings'
os.makedirs(model_save_dir, exist_ok=True)

task_embedding_model.save(os.path.join(model_save_dir, 'task_gcn_embedding_model_3d.keras'))
disorder_embedding_model.save(os.path.join(model_save_dir, 'disorder_gcn_embedding_model_3d.keras'))

print("\n3D GNN Embedding models explicitly trained and saved successfully.")

# === Plot the loss curves and save as PDF ===
plt.figure(figsize=(8, 5))
plt.plot(task_history.history['loss'], label='Task Triplet Loss')
plt.plot(disorder_history.history['loss'], label='Disorder Triplet Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
#plt.title('Triplet Loss Curves for Task and Disorder Embeddings')
plt.legend()
plt.tight_layout()

# Save to PDF
plot_dir = '/content/drive/My Drive/AAAI_fMRI_2025/AAAI_Final_ROI_Yeo17Atlas/Figures'
os.makedirs(plot_dir, exist_ok=True)
loss_plot_path = os.path.join(plot_dir, 'Triplet_Embedding_Loss_Curves_v2.pdf')
plt.savefig(loss_plot_path)
plt.close()

print(f"Triplet loss curve saved to: {loss_plot_path}")


In [ ]:
## 7.proposed Model : ONLY MTL
# New Model Attempt 2:  Weighted OcREAD (Brain Network Transformer) + Dynamic ROi  + Triplet embeddings (pre-traiened model)-----Best Model so far
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import (Input, Dense, Dropout, LayerNormalization, MultiHeadAttention,
                                     Add, Layer, Flatten, Concatenate)
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
import pickle
import networkx as nx
from glob import glob
import os

np.random.seed(42)
tf.random.set_seed(42)


# Step 3: Load adjacency matrices explicitly from graphs and augmented graphs
base_dir = '/content/drive/My Drive/AAAI_fMRI_2025/AAAI_Final_ROI_Yeo17Atlas'

ADHD_dirs = [
    f'{base_dir}/Graphs/ADHD',
    f'{base_dir}/Graph_Augmentation/ADHD'
]

HC_dirs = [
    f'{base_dir}/Graphs/HC',
    f'{base_dir}/Graph_Augmentation/HC'
]

tasks = ['SLD', 'SSD', 'SSI', 'VLD']

def load_graph(filepath):
    with open(filepath, 'rb') as f:
        return pickle.load(f)

def load_data(dirs):
    adj_matrices, task_labels, disorder_labels = [], [], []
    for d in dirs:
        files = sorted(glob(os.path.join(d, '*.pkl')))
        for file in files:
            graph = load_graph(file)
            adj_matrix = nx.to_numpy_array(graph)
            adj_matrices.append(adj_matrix)

            fname = os.path.basename(file)
            task_label = tasks.index(fname.split('_')[1])
            disorder_label = 0 if 'ADHD' in fname else 1

            task_labels.append(task_label)
            disorder_labels.append(disorder_label)

    return np.array(adj_matrices), np.array(task_labels), np.array(disorder_labels)

X_adj, y_task, y_disorder = load_data(ADHD_dirs + HC_dirs)
print("Adjacency matrices shape:", X_adj.shape)

# Step 4: Load pre-trained embedding models explicitly defined
embedding_dir = f'{base_dir}/GNN_Embeddings'

# Explicitly load trained embedding models (now without errors explicitly)
# task_embedding_model = load_model(f'{embedding_dir}/task_gcn_embedding_model_3d.keras', compile=False)
# disorder_embedding_model = load_model(f'{embedding_dir}/disorder_gcn_embedding_model_3d.keras', compile=False)

# Load trained embedding models explicitly
# Explicit custom layer definitions
@tf.keras.utils.register_keras_serializable()
class NodeFeatureLayer(Layer):
    def call(self, adj):
        num_nodes = tf.shape(adj)[-1]
        batch_size = tf.shape(adj)[0]
        return tf.eye(num_nodes, batch_shape=[batch_size])

@tf.keras.utils.register_keras_serializable()
class GraphConvLayer(Layer):
    def __init__(self, output_dim, activation='relu', **kwargs):
        super().__init__(**kwargs)
        self.output_dim = output_dim
        self.activation = tf.keras.activations.get(activation)

    def build(self, input_shape):
        feature_dim = input_shape[0][-1]
        self.weight = self.add_weight(shape=(feature_dim, self.output_dim),
                                      initializer='glorot_uniform',
                                      trainable=True)

    def call(self, inputs):
        x, adj = inputs
        h = tf.matmul(x, self.weight)
        output = tf.matmul(adj, h)
        return self.activation(output)

@tf.keras.utils.register_keras_serializable()
class GlobalMeanPoolLayer(Layer):
    def call(self, inputs):
        return tf.reduce_mean(inputs, axis=1)


task_embedding_model = load_model(
    os.path.join(embedding_dir, 'task_gcn_embedding_model_3d.keras'), compile=False,
    custom_objects={'NodeFeatureLayer': NodeFeatureLayer,
                    'GraphConvLayer': GraphConvLayer,
                    'GlobalMeanPoolLayer': GlobalMeanPoolLayer}
)

disorder_embedding_model = load_model(
    os.path.join(embedding_dir, 'disorder_gcn_embedding_model_3d.keras'), compile=False,
    custom_objects={'NodeFeatureLayer': NodeFeatureLayer,
                    'GraphConvLayer': GraphConvLayer,
                    'GlobalMeanPoolLayer': GlobalMeanPoolLayer}
)

# Generate embeddings explicitly
task_embeddings = task_embedding_model.predict(X_adj, verbose=0)
disorder_embeddings = disorder_embedding_model.predict(X_adj, verbose=0)

print("Task embeddings shape:", task_embeddings.shape)
print("Disorder embeddings shape:", disorder_embeddings.shape)

# Step 5: Train-Test Split explicitly defined
(X_adj_train, X_adj_test,
 X_task_embed_train, X_task_embed_test,
 X_dis_embed_train, X_dis_embed_test,
 y_task_train, y_task_test,
 y_dis_train, y_dis_test) = train_test_split(
    X_adj, task_embeddings, disorder_embeddings, y_task, y_disorder,
    test_size=0.2, stratify=y_task, random_state=42)

print("Train adjacency shape:", X_adj_train.shape)
print("Train task embeddings shape:", X_task_embed_train.shape)
print("Train disorder embeddings shape:", X_dis_embed_train.shape)

# Step 6: Define Multi-head Self-Attention (MHSA) block explicitly defined
def mhsa_block(x, num_heads=8, key_dim=64, dropout=0.3):
    attn_out = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(x, x)
    attn_out = Dropout(dropout)(attn_out)
    x = Add()([x, attn_out])
    x = LayerNormalization()(x)
    return x

# Step 7: Dynamic ROI Attention explicitly defined
class DynamicROIAttention(Layer):
    def build(self, input_shape):
        self.attn_dense = Dense(input_shape[-1], activation='relu')
        self.attn_vector = Dense(1, activation='softmax')

    def call(self, inputs):
        attn_scores = self.attn_dense(inputs)
        attn_scores = self.attn_vector(attn_scores)
        attended = inputs * attn_scores
        return attended

# Step 8: Weighted OCREAD explicitly defined
class WeightedOCREAD(Layer):
    def __init__(self, num_clusters, embedding_dim):
        super().__init__()
        self.num_clusters = num_clusters
        self.embedding_dim = embedding_dim

    def build(self, input_shape):
        self.cluster_centers = self.add_weight(shape=(self.num_clusters, input_shape[-1]),
                                               initializer='orthogonal', trainable=True)

    def call(self, inputs, adjacency_matrix):
        assignment_logits = tf.matmul(inputs, self.cluster_centers, transpose_b=True)
        assignment_probs = tf.nn.softmax(assignment_logits, axis=-1)
        connectivity_strength = tf.reduce_mean(adjacency_matrix, axis=-1, keepdims=True)
        weighted_inputs = inputs * connectivity_strength
        embedding = tf.matmul(assignment_probs, weighted_inputs, transpose_a=True)
        return Flatten()(embedding)

# Step 9: Explicitly Defined Brain Transformer MTL Model with embeddings
def build_brain_transformer_mtl(input_shape_adj, input_shape_task_embed, input_shape_dis_embed,
                                num_task_clusters=4, num_disorder_clusters=2):
    adj_input = Input(shape=input_shape_adj)
    task_embed_input = Input(shape=input_shape_task_embed)
    disorder_embed_input = Input(shape=input_shape_dis_embed)

    # MHSA blocks explicitly defined for adjacency matrices
    x = mhsa_block(adj_input, num_heads=8, key_dim=64, dropout=0.3)
    x = mhsa_block(x, num_heads=8, key_dim=64, dropout=0.3)
    #x = mhsa_block(x, num_heads=8, key_dim=64, dropout=0.2)

    # Dynamic ROI Attention explicitly defined
    x_dynamic_roi = DynamicROIAttention()(x)

    # Task-specific Weighted OCREAD explicitly defined (explicitly flattened already)
    task_ocread = WeightedOCREAD(num_clusters=num_task_clusters, embedding_dim=68)(x_dynamic_roi, adj_input)

    # Explicitly Pool or Flatten Triplet Embeddings for Tasks
    task_embed_flat = Flatten()(task_embed_input)  # Now shape explicitly (None, 17*64=1088)

    # Concatenate explicitly flattened triplet embedding explicitly with OCREAD embedding
    task_combined = Concatenate(axis=-1)([task_ocread, task_embed_flat])
    task_branch = Dense(164, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(1e-3))(task_combined)
    task_branch = Dropout(0.4)(task_branch)
    # task_branch = Dense(48, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(1e-3))(task_branch)
    # task_branch = Dropout(0.4)(task_branch)
    task_output = Dense(4, activation='softmax', name='task_output')(task_branch)

    # Disorder-specific Weighted OCREAD explicitly defined (explicitly flattened)
    disorder_ocread = WeightedOCREAD(num_clusters=num_disorder_clusters, embedding_dim=34)(x_dynamic_roi, adj_input)

    # Explicitly Flatten Triplet Embeddings for Disorders
    disorder_embed_flat = Flatten()(disorder_embed_input)  # shape explicitly (None, 17*64=1088)

    # Concatenate explicitly flattened disorder triplet embedding explicitly with OCREAD embedding explicitly
    disorder_combined = Concatenate(axis=-1)([disorder_ocread, disorder_embed_flat])
    disorder_branch = Dense(164, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(1e-3))(disorder_combined)
    disorder_branch = Dropout(0.4)(disorder_branch)
    disorder_branch = Dense(32, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(1e-3))(disorder_branch)
    disorder_branch = Dropout(0.4)(disorder_branch)
    disorder_output = Dense(1, activation='sigmoid', name='disorder_output')(disorder_branch)

    # Final Explicitly defined MTL model explicitly
    model = Model(inputs=[adj_input, task_embed_input, disorder_embed_input],
                  outputs=[task_output, disorder_output])

    model.compile(optimizer=Adam(learning_rate=5e-4), #5e-4
                  loss={'task_output': 'sparse_categorical_crossentropy',
                        'disorder_output': 'binary_crossentropy'},
                  metrics={'task_output': 'accuracy', 'disorder_output': 'accuracy'})
    return model

model = build_brain_transformer_mtl(X_adj_train.shape[1:], X_task_embed_train.shape[1:], X_dis_embed_train.shape[1:])
model.summary()

####Step 10: Model training explicitly
callbacks=[
    EarlyStopping(patience=40, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', patience=20, factor=0.5, min_lr=1e-7, verbose=1)
]

history = model.fit(
    [X_adj_train, X_task_embed_train, X_dis_embed_train],
    {'task_output': y_task_train, 'disorder_output': y_dis_train},
    epochs=100, batch_size=128, validation_split=0.1,
    callbacks=callbacks
    )

## Save the model
# === Save the trained MTL model
model_save_path = os.path.join(base_dir, 'brain_transformer_mtl_model.h5')
model.save(model_save_path)
print(f"Model saved to: {model_save_path}")

# Step 11: Explicit evaluation
task_pred, dis_pred = model.predict([X_adj_test, X_task_embed_test, X_dis_embed_test])
task_labels_pred = np.argmax(task_pred, axis=1)
dis_labels_pred = (dis_pred > 0.5).astype(int).flatten()

print("\nTask Accuracy:", accuracy_score(y_task_test, task_labels_pred))
print(classification_report(y_task_test, task_labels_pred, target_names=tasks))

print("\nDisorder Accuracy:", accuracy_score(y_dis_test, dis_labels_pred))
print(classification_report(y_dis_test, dis_labels_pred, target_names=['ADHD','HC']))

## plots
plt.figure(figsize=(12,5))

# Task Loss explicitly
plt.subplot(1,2,1)
plt.plot(history.history['task_output_loss'], label='Task Train Loss')
plt.plot(history.history['val_task_output_loss'], label='Task Val Loss')
plt.title('Task Classification Loss')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend()

# Disorder Loss explicitly
plt.subplot(1,2,2)
plt.plot(history.history['disorder_output_loss'], label='Disorder Train Loss')
plt.plot(history.history['val_disorder_output_loss'], label='Disorder Val Loss')
plt.title('Disorder Classification Loss')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend()

plt.tight_layout()
plt.show()